In [1]:
year = 1995
month = 11

In [2]:
import numpy as np
import pandas as pd
import xarray as xr
import os, pathlib, stat, textwrap

### URLs

In [3]:
# Ufiles = "https://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Ufiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Vfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridV"
Wfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridW"
Tfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridT"
Sfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridS"
#mesh url
Zgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_zgr.nc"
Hgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_hgr.nc"

### Environment 

In [4]:
os.environ["NETRC"] = "/home/b/b383184/.netrc"

### Mesh

In [5]:
ds_Zgr = xr.open_dataset('../data/Zgr_mesh.nc')
ds_Zgr

<xarray.Dataset> Size: 555MB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/13)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    mbathy        (t, y, x) int16 26MB ...
    hdept         (t, y, x) float64 106MB ...
    ...            ...
    e3t_ps        (t, y, x) float64 106MB ...
    e3w_ps        (t, y, x) float64 106MB ...
    gdept_0       (t, z) float64 400B ...
    gdepw_0       (t, z) float64 400B ...
    e3t_0         (t, z) float64 400B ...
    e3w_0         (t, z) float64 400B ...
Attributes:
    file_name:            mesh_zgr.nc
    TimeStamp:            03/02/2016 10:24:41 -0000
    Unlimited_Dimension:  t

In [6]:
ds_Hgr = xr.open_dataset('../data/Hgr_mesh.nc')
ds_Hgr

<xarray.Dataset> Size: 1GB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/21)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    glamt         (t, y, x) float32 53MB ...
    glamu         (t, y, x) float32 53MB ...
    ...            ...
    e1f           (t, y, x) float64 106MB ...
    e2t           (t, y, x) float64 106MB ...
    e2u           (t, y, x) float64 106MB ...
    e2v           (t, y, x) float64 106MB ...
    e2f           (t, y, x) float64 106MB ...
    ff            (t, y, x) float64 106MB ...
Attributes:
    file_name:            mesh_hgr.nc
    TimeStamp:            03/02/2016 10:24:55 -0000
    Unlimited_Dimension:  t

### Functions

In [7]:
lon0, lon1 = -95, 10
lat0, lat1 = -10, 30

# 1) Use T-point lon/lat
lonT = ds_Hgr.glamt.isel(t=0)
latT = ds_Hgr.gphit.isel(t=0)

# 2) Build boolean mask for your box
mask = (lonT >= lon0) & (lonT <= lon1) & (latT >= lat0) & (latT <= lat1)

# 3) Get index ranges
yy, xx = np.where(mask.values)

y0, y1 = int(yy.min()), int(yy.max())
x0, x1 = int(xx.min()), int(xx.max())

x0, x1, y0, y1

# x0=int(ds_Hgr.x.where(ds_Hgr.glamt >= lon0).min())
# x1=int(ds_Hgr.x.where(ds_Hgr.glamt <= lon1).max())

# y0=int(ds_Hgr.y.where(ds_Hgr.gphit >= lat0).min())
# y1=int(ds_Hgr.y.where(ds_Hgr.gphit <= lat1).max())

(2305, 3565, 1374, 1873)

In [8]:
# ds_Zgr.isel(x=slice(x0,x1),y=slice(y0,y1),t=0).mbathy.plot()

In [9]:
import calendar
import datetime
from datetime import date

In [10]:
last_day = calendar.monthrange(year, month)[1]
start_date = datetime.datetime(year, month, 1)
end_date = datetime.datetime(year, month, last_day)

print(end_date.strftime("%Y-%m-%d"))

1995-11-30


In [11]:
import pandas as pd

def glorys_days(start_date, end_date):
    return pd.date_range(start=start_date, end=end_date, freq="D") + pd.Timedelta(hours=12)

# Example
days = glorys_days(start_date.strftime("%Y-%m-%d")
                   , end_date.strftime("%Y-%m-%d"))

In [12]:
def download_MERCATOR(url, varname, starts, ends, x0, x1, y0, y1, output_file):

    from tqdm import tqdm
    import xarray as xr
    
    parts = []
    for tt in tqdm(range(len(days)//2)):
        da = (
            xr.open_dataset(url, engine="pydap", mask_and_scale=False, decode_cf=True)[varname]
            .sortby("time_counter")
            .isel(x=slice(x0, x1), y=slice(y0, y1))
            .sel(time_counter=slice(starts[tt], ends[tt]))
            .astype("float32")
            .load()
        )
        parts.append(da)
    
    da_all = xr.concat(parts, dim="time_counter")
    da_all.to_dataset(name=varname).to_netcdf(output_file, unlimited_dims=["time_counter"])
    print(f"Saved {output_file}")

In [13]:
# import numpy as np
# a=np.arange(0,len(days),2)
# b=np.arange(0+1,len(days),2)
# print(a)
# print(b)
# starts = days[a]
# ends = days[b]
starts = days[0::2]
ends = days[1::2].tolist()  
# ends = days[b].tolist()  
ends[-1] = days[-1]
ends

for tt in  range(len(days)//2):
    print('start_date '+str(starts[tt]))
    print('end_date '+str(ends[tt]))

start_date 1995-11-01 12:00:00
end_date 1995-11-02 12:00:00
start_date 1995-11-03 12:00:00
end_date 1995-11-04 12:00:00
start_date 1995-11-05 12:00:00
end_date 1995-11-06 12:00:00
start_date 1995-11-07 12:00:00
end_date 1995-11-08 12:00:00
start_date 1995-11-09 12:00:00
end_date 1995-11-10 12:00:00
start_date 1995-11-11 12:00:00
end_date 1995-11-12 12:00:00
start_date 1995-11-13 12:00:00
end_date 1995-11-14 12:00:00
start_date 1995-11-15 12:00:00
end_date 1995-11-16 12:00:00
start_date 1995-11-17 12:00:00
end_date 1995-11-18 12:00:00
start_date 1995-11-19 12:00:00
end_date 1995-11-20 12:00:00
start_date 1995-11-21 12:00:00
end_date 1995-11-22 12:00:00
start_date 1995-11-23 12:00:00
end_date 1995-11-24 12:00:00
start_date 1995-11-25 12:00:00
end_date 1995-11-26 12:00:00
start_date 1995-11-27 12:00:00
end_date 1995-11-28 12:00:00
start_date 1995-11-29 12:00:00
end_date 1995-11-30 12:00:00


### Cut the meshes

In [21]:
!pwd

/work/bk1450/b383184/Amazon/Mercator/notebooks


In [20]:
# nZGR = ds_Zgr.isel(x=slice(x0, x1), y=slice(y0, y1))
# nZGR.to_netcdf('Zgr_cmesh')

In [23]:
# nHgr = ds_Hgr.isel(x=slice(x0, x1), y=slice(y0, y1))
# nHgr.to_netcdf('../data/Hgr_cmesh.nc')

### Data download

In [14]:
U_out = f'U_{start_date.strftime("%Y-%m-%d")[:7]}.nc'
V_out = f'V_{start_date.strftime("%Y-%m-%d")[:7]}.nc'
W_out = f'W_{start_date.strftime("%Y-%m-%d")[:7]}.nc'
T_out = f'T_{start_date.strftime("%Y-%m-%d")[:7]}.nc'
S_out = f'S_{start_date.strftime("%Y-%m-%d")[:7]}.nc'

outpath = '/work/bk1450/b383184/Amazon/Mercator/data/variables/'

In [15]:

download_MERCATOR(
    Ufiles, "vozocrtx", starts, ends, x0, x1, y0, y1,outpath+U_out
)

In [16]:
download_MERCATOR(
    Vfiles, "vomecrty", starts, ends, x0, x1, y0, y1,outpath+V_out 
)

In [17]:
download_MERCATOR(
    Wfiles, "vovecrtz", starts, ends, x0, x1, y0, y1,outpath+W_out 
)

In [18]:
download_MERCATOR(
    Tfiles, "votemper", starts, ends, x0, x1, y0, y1,outpath+T_out
)

In [19]:
download_MERCATOR(
    Sfiles, "vosaline", starts, ends, x0, x1, y0, y1,outpath+S_out 
)

100%|██████████| 15/15 [07:05<00:00, 28.40s/it]


Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/S_1995-11.nc


In [20]:
# ds_U = (
#     xr.open_dataset(Ufiles, engine="pydap")
#     .sortby("time_counter").isel(x=slice(x0,x1),y=slice(y0,y1))
#     .sel(time_counter=slice('2004-02-01','2004-02-02'))
#     .vozocrtx
# )
# ds_U

In [21]:
# ds_U = open_subset_var(Ufiles, "vozocrtx", x0, x1+1, y0, y1+1, start_date, end_date, engine="pydap")
# ds_V = open_subset_var(Vfiles, "vomecrty", x0, x1+1, y0, y1+1, start_date, end_date, engine="pydap")
# ds_W = open_subset_var(Wfiles, "vovecrtz", x0, x1+1, y0, y1+1, start_date, end_date, engine="pydap")
# ds_T = open_subset_var(Tfiles, "votemper", x0, x1+1, y0, y1+1, start_date, end_date, engine="pydap")
# ds_S = open_subset_var(Sfiles, "vosaline", x0, x1+1, y0, y1+1, start_date, end_date, engine="pydap")

In [22]:
# from tqdm import tqdm

# parts = []
# for tt in  tqdm(range(len(days)//2)):
#     U = (
#         xr.open_dataset(Ufiles, engine="pydap",
#                         mask_and_scale=False, decode_cf=True)["vozocrtx"]
#         .sortby("time_counter").isel(x=slice(x0,x1),y=slice(y0,y1))
#         .sel(time_counter=slice(starts[tt],ends[tt]))
#         .astype("float32")
#         .load()
#     )
#     parts.append(U)

# U_all = xr.concat(parts, dim="time_counter")
# U_all.to_dataset(name="vozocrtx").to_netcdf(f'U_{start_date.strftime("%Y-%m-%d")[:7]}.nc')

In [28]:
import xarray as xr
aa = xr.open_dataset('U_2007-01.nc')
print(aa)

<xarray.Dataset> Size: 8GB
Dimensions:       (deptht: 50, x: 1260, y: 499, time_counter: 31)
Coordinates:
  * deptht        (deptht) float32 200B 0.494 1.541 ... 5.275e+03 5.728e+03
  * x             (x) int32 5kB 2306 2307 2308 2309 2310 ... 3562 3563 3564 3565
  * y             (y) int32 2kB 1375 1376 1377 1378 1379 ... 1870 1871 1872 1873
  * time_counter  (time_counter) datetime64[ns] 248B 2007-01-01T12:00:00 ... ...
    nav_lon       (y, x) float32 3MB ...
    nav_lat       (y, x) float32 3MB ...
Data variables:
    vozocrtx      (time_counter, deptht, y, x) float64 8GB ...


/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'vozocrtx' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


In [29]:
import xarray as xr
aa = xr.open_dataset('../data/variables/W_1997-01.nc')
print(aa) 

<xarray.Dataset> Size: 8GB
Dimensions:       (depthw: 50, x: 1260, y: 499, time_counter: 31)
Coordinates:
  * depthw        (depthw) float32 200B 0.0 1.011 2.086 ... 5.052e+03 5.5e+03
  * x             (x) int32 5kB 2306 2307 2308 2309 2310 ... 3562 3563 3564 3565
  * y             (y) int32 2kB 1375 1376 1377 1378 1379 ... 1870 1871 1872 1873
  * time_counter  (time_counter) datetime64[ns] 248B 1997-01-01T12:00:00 ... ...
    nav_lon       (y, x) float32 3MB ...
    nav_lat       (y, x) float32 3MB ...
Data variables:
    vovecrtz      (time_counter, depthw, y, x) float64 8GB ...


/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'vovecrtz' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


In [25]:
aa = xr.open_dataset('../data/variables/S_1997-01.nc')
aa 

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'vosaline' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


<xarray.Dataset> Size: 8GB
Dimensions:       (deptht: 50, x: 1260, y: 499, time_counter: 31)
Coordinates:
  * deptht        (deptht) float32 200B 0.494 1.541 ... 5.275e+03 5.728e+03
  * x             (x) int32 5kB 2306 2307 2308 2309 2310 ... 3562 3563 3564 3565
  * y             (y) int32 2kB 1375 1376 1377 1378 1379 ... 1870 1871 1872 1873
  * time_counter  (time_counter) datetime64[ns] 248B 1997-01-01T12:00:00 ... ...
    nav_lon       (y, x) float32 3MB ...
    nav_lat       (y, x) float32 3MB ...
Data variables:
    vosaline      (time_counter, deptht, y, x) float64 8GB ...

In [27]:
ds_Zgr = xr.open_dataset('../data/Zgr_cmesh2.nc')
ds_Zgr

<xarray.Dataset> Size: 26MB
Dimensions:       (y: 499, x: 1260, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/13)
    nav_lon       (y, x) float32 3MB ...
    nav_lat       (y, x) float32 3MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    mbathy        (t, y, x) int16 1MB ...
    deptht        (t, y, x) float64 5MB ...
    ...            ...
    e3t_ps        (t, y, x) float64 5MB ...
    e3w_ps        (t, y, x) float64 5MB ...
    gdept_0       (t, z) float64 400B ...
    gdepw_0       (t, z) float64 400B ...
    e3t_0         (t, z) float64 400B ...
    e3w_0         (t, z) float64 400B ...
Attributes:
    file_name:            mesh_zgr.nc
    TimeStamp:            03/02/2016 10:24:41 -0000
    Unlimited_Dimension:  t
    history:              Wed Nov  5 17:28:38 2025: ncrename -v hdept,deptht ...
    NCO:                  netCDF Operators version 5.0.6 (Homepage = http://n...

In [24]:
# aa.where(aa.vozocrtx!= 9.96921e+36).vozocrtx.isel(time_counter=0,depth=0).plot(robust=True)

In [25]:
# ds_U.isel(time_counter=0,deptht=0).plot()

In [26]:
# ds_V = xr.open_dataset(Vfiles, engine="pydap").sortby("time_counter").isel(x=slice(x0,x1),y=slice(y0,y1)).sel(time_counter=slice(start_date,end_date)).vomecrty
# ds_V

In [27]:
# ds_W = xr.open_dataset(Wfiles, engine="pydap").sortby("time_counter").isel(x=slice(x0,x1),y=slice(y0,y1)).sel(time_counter=slice(start_date,end_date)).vovecrtz
# ds_W

In [28]:
# ds_T = xr.open_dataset(Tfiles, engine="pydap").sortby("time_counter").isel(x=slice(x0,x1),y=slice(y0,y1)).sel(time_counter=slice(start_date,end_date)).votemper
# ds_T

In [29]:
# ds_S = xr.open_dataset(Sfiles, engine="pydap").sortby("time_counter").isel(x=slice(x0,x1),y=slice(y0,y1)).sel(time_counter=slice(start_date,end_date)).vosaline
# ds_S

In [30]:
# ds_S.isel(time_counter=0,deptht=0).where(ds_S.isel(time_counter=0,deptht=0) !=9.96920997e+36,np.nan).plot() 

In [31]:
# da = ds_S
# fill = getattr(da, "_FillValue", 9.969209968386869e+36)
# da = da.where(da != fill)  # keep valid values, others become NaN
# da.plot()

In [32]:
# ds_S.nbytes / 1024**3  # GB actually held in memory